# Preliminary PPMI audit

This notebook rebuilds the data audit behind our Neuromatch Impact Scholars proposal (group NeuroNexus, 2026). It covers which imaging modalities are longitudinal, how grey-matter volume (GMV) overlaps with complete cognition and DaTscan, and how many participants contribute repeated measurements to each domain. It reads the downloads described in `data.ipynb`.

PPMI keeps adding visits, so a newer download gives larger counts than our proposal, but every table has the same structure. The numbers quoted in the text come from our downloads of September to November 2026.

In [ ]:
%matplotlib inline
import sys
sys.path.append("../code")

import numpy as np
import pandas as pd

import data
import plot

p = data.PPMI()

## 1. Which imaging modalities are longitudinal?

I searched IDA for participants with Parkinson's disease who had SPECT (DaTscan), 3D T1-weighted MRI or diffusion MRI (DTI) at screening, baseline and months 12 to 60, and counted unique participants rather than images. In our search, SPECT had 1,638 participants at screening, 177 at baseline, 911 at month 12, 955 at month 24, 1 at month 36, 518 at month 48 and none at month 60. DTI had 664 at baseline, 430 at month 12, 267 at month 24 and 144 at month 48.

In [ ]:
by_visit = p.sql("SELECT modality, visit, event, count(DISTINCT PATNO) AS participants FROM images GROUP BY ALL")
plot.availability(by_visit)

Months 36 and 60 add almost nothing, so each modality has a main series of four visits. For SPECT the series is screening and months 12, 24 and 48, because the baseline SPECT participants barely overlap the screening ones. For MRI and DTI it is baseline and months 12, 24 and 48. In our search 1,133 participants had SPECT at two or more of these visits, 859 at three and 349 at all four. For T1 MRI the counts were about 327, 207 and 92, and for DTI 442, 249 and 91.

In [ ]:
p.sql("""
    WITH series AS (
        SELECT modality, PATNO, count(DISTINCT event) AS visits
        FROM images
        WHERE event IN (CASE modality WHEN 'SPECT' THEN 'SC' ELSE 'BL' END, 'V04', 'V06', 'V10')
        GROUP BY ALL
    )
    SELECT modality,
           count(*) FILTER (visits >= 2) AS "2+ visits",
           count(*) FILTER (visits >= 3) AS "3+ visits",
           count(*) FILTER (visits = 4) AS "all 4"
    FROM series GROUP BY modality ORDER BY modality
""")

## 2. Processed structural MRI

PPMI also shares measures derived from T1 MRI: FreeSurfer regional volumes (`FS7_ASEG_VOL`), cortical thickness (`FS7_APARC_CTH`) and total grey-matter volume (`Grey_Matter_Volume`). In our download only GMV was repeated, at baseline (BL), month 24 (V06) and month 48 (V10). The FreeSurfer tables had baseline only, so regional change over time would mean processing the raw scans ourselves.

In [ ]:
derived = p.sql(" UNION ALL ".join(
    f"SELECT '{name}' AS source, EVENT_ID, count(DISTINCT PATNO) AS participants FROM {name} GROUP BY ALL"
    for name in ["fs7_aseg_vol", "fs7_aparc_cth", "grey_matter_volume"]
    if name in p.files
))
plot.grouped(derived.pivot(index="EVENT_ID", columns="source", values="participants"), "Participants with processed MRI measures, by visit")

## 3. The longitudinal GMV cohort

The GMV table names participants and visits but not diagnosis, so I join it to `Participant_Status` for the cohort. In our download GMV covered 132 participants at baseline, 126 at month 24 and 105 at month 48. All 137 participants with GMV at two or more visits had Parkinson's disease, and 89 of them had all three visits.

In [ ]:
p.sql("""
    WITH gmv AS (
        SELECT PATNO, count(DISTINCT EVENT_ID) AS visits,
               count(DISTINCT EVENT_ID) FILTER (EVENT_ID IN ('BL', 'V06', 'V10')) = 3 AS all_three
        FROM grey_matter_volume WHERE num(GM_VOLUME) > 0 GROUP BY PATNO
    )
    SELECT s.COHORT_DEFINITION AS cohort, count(*) AS participants,
           count(*) FILTER (visits >= 2) AS "2+ visits",
           count(*) FILTER (all_three) AS "BL, V06 and V10"
    FROM gmv LEFT JOIN participant_status s USING (PATNO)
    GROUP BY ALL ORDER BY participants DESC
""")

## 4. Complete cognition at the GMV visits

I matched GMV and cognition on both `PATNO` and `EVENT_ID`, so a match means the scan and the tests come from the same visit, and a visit counts only when all six CSS tests are present. In our download almost nothing was lost: 132 of 132 at baseline, 125 of 126 at month 24 and 103 of 105 at month 48.

In [ ]:
p.db.execute("""
    CREATE OR REPLACE TABLE structural AS
    SELECT DISTINCT g.PATNO, g.EVENT_ID, c.PATNO IS NOT NULL AS complete
    FROM grey_matter_volume g
    LEFT JOIN (
        SELECT PATNO, EVENT_ID FROM cognition WHERE score IS NOT NULL
        GROUP BY ALL HAVING count(DISTINCT test) = 6
    ) c USING (PATNO, EVENT_ID)
    WHERE num(g.GM_VOLUME) > 0
""")
matched = p.sql("""
    SELECT EVENT_ID, count(*) AS GMV, count(*) FILTER (complete) AS "GMV and all six tests"
    FROM structural GROUP BY ALL ORDER BY ALL
""")
plot.grouped(matched.set_index("EVENT_ID"), "GMV visits with complete cognition")

In our download 136 participants had two or more matched visits, the cohort for within-person change, and 87 had all three, the complete-case cohort. These are the combinations:

In [ ]:
combinations = p.sql("""
    SELECT visits, count(*) AS participants FROM (
        SELECT PATNO, string_agg(EVENT_ID, ' + ' ORDER BY EVENT_ID) AS visits
        FROM structural WHERE complete GROUP BY PATNO
    ) GROUP BY visits ORDER BY participants DESC
""")
plot.counts(combinations.set_index("visits").participants, "Visits with GMV and complete cognition")

## 5. Do they also have longitudinal DaTscan?

The two-process model needs dopaminergic and structural measurements in the same people. For participants with two or more matched visits, I counted quantified DaTscans in the main SPECT series (screening and months 12, 24 and 48). DaTscan and MRI follow different schedules, so I also paired screening DaTscan with baseline MRI and months 24 and 48 with themselves. In our download 132 of the 136 had DaTscan at two or more of these visits. 125 had at least two aligned pairs and 78 had all three.

In [ ]:
overlap = p.sql("""
    WITH cohort AS (SELECT PATNO FROM structural WHERE complete GROUP BY PATNO HAVING count(*) >= 2),
    dat AS (
        SELECT DISTINCT PATNO, EVENT_ID FROM xing_core_lab_quant_sbr
        WHERE upper(DATSCAN_ANALYZED) = 'YES' AND EVENT_ID IN ('SC', 'V04', 'V06', 'V10')
          AND greatest(num(CAUDATE_L_REF_CWM), num(CAUDATE_R_REF_CWM), num(PUTAMEN_L_REF_CWM), num(PUTAMEN_R_REF_CWM)) > 0
    ),
    visits AS (SELECT PATNO, count(d.EVENT_ID) AS n FROM cohort LEFT JOIN dat d USING (PATNO) GROUP BY PATNO),
    pairs AS (
        SELECT s.PATNO, count(*) AS n
        FROM structural s
        JOIN cohort USING (PATNO)
        JOIN dat d ON d.PATNO = s.PATNO AND d.EVENT_ID = CASE s.EVENT_ID WHEN 'BL' THEN 'SC' ELSE s.EVENT_ID END
        WHERE s.complete AND s.EVENT_ID IN ('BL', 'V06', 'V10')
        GROUP BY s.PATNO
    )
    SELECT
        (SELECT count(*) FROM cohort) AS "2+ visits with GMV and cognition",
        (SELECT count(*) FILTER (n >= 1) FROM visits) AS "  and DaTscan at 1+ main visits",
        (SELECT count(*) FILTER (n >= 2) FROM visits) AS "  and DaTscan at 2+ main visits",
        (SELECT count(*) FILTER (n >= 3) FROM visits) AS "  and DaTscan at 3+ main visits",
        (SELECT count(*) FILTER (n = 4) FROM visits) AS "  and DaTscan at all 4 main visits",
        (SELECT count(*) FILTER (n >= 2) FROM pairs) AS "  with 2+ aligned DaTscan-MRI pairs",
        (SELECT count(*) FILTER (n = 3) FROM pairs) AS "  with all 3 aligned pairs"
""").T[0]
plot.counts(overlap, "DaTscan in the structural cohort")

## 6. The audit in our proposal

The proposal counts participants who can contribute repeated measurements to each domain, with stricter rules than the sections above:

- Parkinson's disease cohort, enrolled, withdrawn or completed, and untreated at enrolment (`PDTRTMNT` = 0 on the screening or baseline MDS-UPDRS Part III).
- Screening and baseline count as one visit.
- Each measurement is placed in years from the participant's baseline using its actual date, or the nominal visit schedule when the date is missing. Only measurements from half a year before to 5.5 years after baseline count.
- Repeated means two or more visits. For cognition a visit counts only when all six tests can be scored against the published norms, which needs age, sex and education.

With these rules our download gave 1,112 participants with repeated complete cognition, 865 with repeated DaTscan, 137 with repeated GMV and 82 with all three.

The norms are from Weintraub et al. (2025), Supplementary Table 1. Each test was regressed on age, sex (1 male, 2 female) and education in robust healthy controls. A score becomes z = (observed − expected) / RMSE, capped at ±3. IDA codes sex as 0 for female and 1 for male, so `2 − SEX` gives the norms' coding.

In [ ]:
norms = pd.DataFrame(
    [
        ("hvlt_immediate", 28.37542, -0.05979, 1.58306, 0.0, 4.32146),
        ("hvlt_delayed", 10.92487, -0.03996, 0.80139, 0.0, 2.12293),
        ("jlo", 15.07467, -0.02374, -1.29422, 0.08678, 1.77892),
        ("lns", 12.30761, -0.05359, 0.0, 0.13650, 2.68988),
        ("semantic", 20.25284, 0.0, 1.71525, 0.0, 5.49634),
        ("sdmt", 76.80621, -0.47289, 0.0, 0.0, 8.95934),
    ],
    columns=["test", "intercept", "age", "sex", "education", "rmse"],
)
p.save("norms", norms)
p.db.execute("""
    CREATE OR REPLACE TABLE scores AS
    WITH sex AS (
        SELECT PATNO, first(2 - num(SEX) ORDER BY EVENT_ID != 'BL') AS sex
        FROM demographics WHERE num(SEX) IN (0, 1) GROUP BY PATNO
    ), education AS (
        SELECT PATNO, first(num(EDUCYRS) ORDER BY EVENT_ID != 'BL') AS education
        FROM socio_economics WHERE num(EDUCYRS) IS NOT NULL GROUP BY PATNO
    ), age AS (
        SELECT PATNO, visit(EVENT_ID) AS event, first(num(AGE_AT_VISIT) ORDER BY EVENT_ID != 'BL') AS age
        FROM age_at_visit WHERE num(AGE_AT_VISIT) IS NOT NULL GROUP BY ALL
    ), tests AS (
        SELECT PATNO, visit(EVENT_ID) AS event, EVENT_ID AS original, date, test, score
        FROM cognition WHERE score IS NOT NULL
        QUALIFY row_number() OVER (PARTITION BY PATNO, visit(EVENT_ID), test ORDER BY EVENT_ID != 'BL', date) = 1
    )
    SELECT t.*, greatest(-3, least(3, (t.score - n.intercept - n.age * a.age - n.sex * s.sex - n.education * e.education) / n.rmse)) AS z
    FROM tests t
    JOIN norms n USING (test)
    JOIN age a USING (PATNO, event)
    JOIN sex s USING (PATNO)
    JOIN education e USING (PATNO)
""")
p.db.execute("""
    CREATE OR REPLACE TABLE eligible AS
    SELECT s.PATNO, ppmi_date(s.ENROLL_DATE) AS enrolled
    FROM participant_status s
    JOIN (
        SELECT PATNO, first(num(PDTRTMNT) ORDER BY num(PDTRTMNT) IS NULL, EVENT_ID != 'BL') AS treated
        FROM mds_updrs_part_iii WHERE EVENT_ID IN ('SC', 'BL') GROUP BY PATNO
    ) u USING (PATNO)
    WHERE num(s.COHORT) = 1 AND lower(s.ENROLL_STATUS) IN ('enrolled', 'withdrew', 'complete') AND u.treated = 0
""")
p.sql("SELECT test, count(DISTINCT PATNO) AS participants, round(avg(z), 2) AS \"mean z\" FROM scores GROUP BY test ORDER BY test")

In [ ]:
p.db.execute("""
    CREATE OR REPLACE TABLE timeline AS
    WITH nominal(original, years) AS (
        VALUES ('SC', 0.0), ('BL', 0.0), ('V02', 0.25), ('V04', 1.0), ('V06', 2.0), ('V08', 3.0), ('V10', 4.0), ('V12', 5.0)
    ), datscan AS (
        SELECT PATNO, visit(EVENT_ID) AS event, EVENT_ID AS original, ppmi_date(DATSCAN_DATE) AS date
        FROM xing_core_lab_quant_sbr
        WHERE upper(DATSCAN_ANALYZED) = 'YES'
          AND greatest(num(CAUDATE_L_REF_CWM), num(CAUDATE_R_REF_CWM), num(PUTAMEN_L_REF_CWM), num(PUTAMEN_R_REF_CWM)) > 0
        QUALIFY row_number() OVER (PARTITION BY PATNO, visit(EVENT_ID) ORDER BY EVENT_ID != 'SC', date) = 1
    ), gmv AS (
        SELECT PATNO, visit(EVENT_ID) AS event, EVENT_ID AS original, ppmi_date(MRIDATE) AS date
        FROM grey_matter_volume WHERE num(GM_VOLUME) > 0
        QUALIFY row_number() OVER (PARTITION BY PATNO, visit(EVENT_ID) ORDER BY date) = 1
    ), observations AS (
        SELECT PATNO, event, original, date, 'cognition' AS domain, test AS marker FROM scores
        UNION ALL SELECT PATNO, event, original, date, 'datscan', 'datscan' FROM datscan
        UNION ALL SELECT PATNO, event, original, date, 'gmv', 'gmv' FROM gmv
    ), anchors AS (
        SELECT e.PATNO, coalesce(CAST(median(o.date) FILTER (o.event = 'BL') AS DATE), any_value(e.enrolled)) AS anchor
        FROM eligible e LEFT JOIN observations o USING (PATNO) GROUP BY e.PATNO
    ), timed AS (
        SELECT o.*, coalesce(date_diff('day', a.anchor, o.date) / 365.25, n.years) AS t
        FROM observations o JOIN anchors a USING (PATNO) LEFT JOIN nominal n USING (original)
    )
    SELECT PATNO, event, domain, marker, CASE WHEN event = 'BL' AND abs(t) < 0.25 THEN 0 ELSE t END AS years
    FROM timed WHERE t BETWEEN -0.5 AND 5.5
""")
audit = p.sql("""
    WITH repeated_cognition AS (
        SELECT PATNO FROM (
            SELECT PATNO, event FROM timeline WHERE domain = 'cognition' GROUP BY ALL HAVING count(DISTINCT marker) = 6
        ) GROUP BY PATNO HAVING count(*) >= 2
    ), repeated_datscan AS (
        SELECT PATNO FROM timeline WHERE domain = 'datscan' GROUP BY PATNO HAVING count(DISTINCT event) >= 2
    ), repeated_gmv AS (
        SELECT PATNO FROM timeline WHERE domain = 'gmv' GROUP BY PATNO HAVING count(DISTINCT event) >= 2
    )
    SELECT
        (SELECT count(*) FROM eligible) AS "untreated PD at enrolment",
        (SELECT count(*) FROM repeated_cognition) AS "repeated complete cognition",
        (SELECT count(*) FROM repeated_datscan) AS "repeated DaTscan",
        (SELECT count(*) FROM repeated_gmv) AS "repeated GMV",
        (SELECT count(*) FROM repeated_cognition JOIN repeated_datscan USING (PATNO) JOIN repeated_gmv USING (PATNO)) AS "all three"
""").T[0]
plot.counts(audit, "Participants with repeated measurements")

## 7. The Cognitive Summary Score

The CSS is the mean of the six z-scores at a visit. Weintraub et al. (2025) showed that at baseline it separates newly diagnosed Parkinson's disease from robust healthy controls better than any single test. Here I compare against every healthy control, at baseline and at each later study year, and give each cohort's pooled yearly change. We reproduced the baseline separation approximately. The difference in yearly change between the groups was small, around −0.03 to −0.04 z per year.

In [ ]:
css = p.sql("""
    SELECT CASE num(s.COHORT) WHEN 1 THEN 'Parkinson''s disease' ELSE 'Healthy control' END AS cohort,
           z.PATNO, z.event, any_value(n.years) AS years, avg(z.z) AS css
    FROM scores z
    JOIN participant_status s USING (PATNO)
    JOIN (VALUES ('BL', 0), ('V04', 1), ('V06', 2), ('V08', 3), ('V10', 4), ('V12', 5)) n(event, years) USING (event)
    WHERE num(s.COHORT) IN (1, 2)
    GROUP BY ALL HAVING count(*) = 6
""")
p.save("css", css)
p.sql("""
    SELECT cohort, count(DISTINCT PATNO) AS participants,
           round(avg(css) FILTER (years = 0), 2) AS "baseline CSS",
           round(regr_slope(css, years), 3) AS "change per year"
    FROM css GROUP BY cohort ORDER BY cohort DESC
""")

In [ ]:
plot.css(css)